# 50.007 ML Course — GenAI Text Detection (Simplified)

**Goal:** Binary classification of text as **human-authored (0)** or **machine-generated (1)**

**Metric:** Macro F1 (both classes weighted equally)

**Best approach:** Custom TF-IDF text representation + Linear SVM
- Validation: **0.8229 macro F1**
- Kaggle Public: **0.72990** (1st place)

---

## Key Insights

1. **Representation > Model:** Fixed 5000 TF-IDF features limit vocabulary. Custom features (640K) win.
2. **Character n-grams capture style:** Punctuation, spacing, suffixes = authorship fingerprint.
3. **Class imbalance matters:** Dataset 62.5% machine / 37.5% human → use `class_weight="balanced"`
4. **Simpler is better:** Linear SVM beats ensembles of correlated feature-based models.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
DATA_DIR = Path("data")
OUTPUT_DIR = Path("predictions")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Libraries loaded")

## 1. Load and Explore Data

In [ ]:
# Load training features (5000 pre-computed TF-IDF features)
train_feat = pd.read_csv(DATA_DIR / "train_features.csv")
test_feat = pd.read_csv(DATA_DIR / "test_features.csv")

# Load raw text (for custom feature engineering)
train_text = pd.read_csv(DATA_DIR / "train.csv")  # id, text, label
test_text = pd.read_csv(DATA_DIR / "test.csv")    # id, text

# Extract features and labels
feature_cols = [c for c in train_feat.columns if c not in ("id", "label")]
X_feat_train = train_feat[feature_cols].to_numpy(dtype=np.float32)
y_train = train_feat["label"].to_numpy(dtype=int)
X_feat_test = test_feat[feature_cols].to_numpy(dtype=np.float32)

X_text_train = train_text["text"].values
X_text_test = test_text["text"].values
test_ids = test_feat["id"].values

print(f"Training data: {X_feat_train.shape}")
print(f"Test data: {X_feat_test.shape}")
print(f"\nClass balance:")
print(f"  Human (0): {(y_train == 0).sum():,} ({(y_train == 0).sum()/len(y_train)*100:.1f}%)")
print(f"  Machine (1): {(y_train == 1).sum():,} ({(y_train == 1).sum()/len(y_train)*100:.1f}%)")

# Show sample texts
print(f"\nSample human text:")
print(f"  {train_text[y_train == 0]['text'].iloc[0][:200]}...")
print(f"\nSample machine text:")
print(f"  {train_text[y_train == 1]['text'].iloc[0][:200]}...")

## 2. Create Train / Validation Split

**Why 10% holdout?** No labeled dev set provided. Stratified split ensures both classes represented.

In [ ]:
tr_idx, val_idx = train_test_split(
    np.arange(len(y_train)), 
    test_size=0.1, 
    stratify=y_train, 
    random_state=SEED
)

print(f"Train split: {len(tr_idx):,} samples")
print(f"Validation split: {len(val_idx):,} samples")

## 3. Baseline: Logistic Regression on Fixed Features

**Algorithm:** Logistic Regression
- Models P(y=1|x) = sigmoid(w·x + b)
- Minimizes: binary cross-entropy loss
- Why? Linear model, interpretable, fast on sparse features
- Best hyperparameters: C=5.0, class_weight="balanced" (to handle 62.5/37.5 imbalance)

In [ ]:
X_tr, y_tr = X_feat_train[tr_idx], y_train[tr_idx]
X_val, y_val = X_feat_train[val_idx], y_train[val_idx]

print("Training Logistic Regression on 5000 features...")
lr = LogisticRegression(
    C=5.0, 
    class_weight="balanced", 
    max_iter=3000, 
    random_state=SEED
)
lr.fit(X_tr, y_tr)

y_val_pred = lr.predict(X_val)
f1_baseline = f1_score(y_val, y_val_pred, average="macro")
acc_baseline = accuracy_score(y_val, y_val_pred)

print(f"\nBASELINE Results:")
print(f"  Validation Macro F1: {f1_baseline:.4f}")
print(f"  Validation Accuracy: {acc_baseline:.4f}")

## 4. Winner: Custom TF-IDF + Linear SVM

### Why this works:

**TF-IDF (Term Frequency - Inverse Document Frequency):**
- Word weights = how unique they are to each class
- Word 1-2grams: captures topical vocabulary
- Character 3-5grams: captures sub-word style (punctuation, spacing, suffixes)
- Result: ~640K features vs 5000 fixed

**Linear SVM (Support Vector Machine):**
- Finds maximum-margin hyperplane separating classes
- Handles high-dimensional sparse input efficiently
- Hinge loss focuses on hard boundary cases
- `class_weight="balanced"`: inverse-weight classes by frequency

In [ ]:
# Build the TF-IDF vectorizer
print("Building custom TF-IDF vectorizer...")

word_vec = TfidfVectorizer(
    ngram_range=(1, 2),       # Unigrams + bigrams
    min_df=2,                 # Ignore words in <2 docs
    sublinear_tf=True,        # Dampen raw term counts: f → 1 + log(f)
    analyzer="word"
)

char_vec = TfidfVectorizer(
    ngram_range=(3, 5),       # 3-character, 4-character, 5-character n-grams
    min_df=2,
    sublinear_tf=True,
    analyzer="char_wb",       # char_wb = preserve word boundaries
    max_features=300_000      # Cap features to avoid memory explosion
)

# Combine word + char features
vectorizer = FeatureUnion([("word", word_vec), ("char", char_vec)])

# Fit on training text
t0 = time.time()
X_text_tr = X_text_train[tr_idx]
X_text_val = X_text_train[val_idx]

X_tr_vec = vectorizer.fit_transform(X_text_tr)
X_val_vec = vectorizer.transform(X_text_val)

print(f"✓ Vectorized: {X_tr_vec.shape[1]:,} features in {time.time()-t0:.1f}s")
print(f"  Train matrix: {X_tr_vec.shape} (sparsity: {100*(1-X_tr_vec.nnz/(X_tr_vec.shape[0]*X_tr_vec.shape[1])):.1f}%)")

### Hyperparameter Tuning

**C:** Regularization parameter (inverse of regularization strength)
- Lower C = stronger regularization (simpler model, less overfitting)
- Higher C = weaker regularization (complex model, more overfitting)

**class_weight:** How to weight minority class
- None: equal weight
- "balanced": weight ∝ 1 / class_frequency

In [ ]:
print("Tuning SVM hyperparameters (C × class_weight):")
results = []

for C in [0.25, 0.5, 1.0, 2.0]:
    for cw in [None, "balanced"]:
        svm = LinearSVC(C=C, class_weight=cw, random_state=SEED, max_iter=2000)
        svm.fit(X_tr_vec, y_tr)
        
        y_pred = svm.predict(X_val_vec)
        f1 = f1_score(y_val, y_pred, average="macro")
        acc = accuracy_score(y_val, y_pred)
        
        results.append({
            "C": C, 
            "class_weight": cw, 
            "Macro F1": f1, 
            "Accuracy": acc
        })
        
        print(f"  C={C:4}, class_weight={str(cw):8} → F1={f1:.4f}, Acc={acc:.4f}")

# Find best
results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
best_row = results_df.iloc[0]
best_C = best_row["C"]
best_cw = best_row["class_weight"]
best_f1_winner = best_row["Macro F1"]

print(f"\n🏆 Best config: C={best_C}, class_weight={best_cw}")
print(f"   Validation Macro F1: {best_f1_winner:.4f}")

### Retrain on Full Dataset & Predict Test Set

In [ ]:
print("Refitting on full 20K training samples...")

# Vectorize full training set
X_train_vec = vectorizer.fit_transform(X_text_train)

# Retrain SVM with best hyperparameters
svm_final = LinearSVC(C=best_C, class_weight=best_cw, random_state=SEED, max_iter=2000)
svm_final.fit(X_train_vec, y_train)

# Predict test set
X_test_vec = vectorizer.transform(X_text_test)
y_test_pred = svm_final.predict(X_test_vec)

# Save predictions
output_file = OUTPUT_DIR / "Winner_Prediction.csv"
df_pred = pd.DataFrame({"id": test_ids, "label": y_test_pred.astype(int)})
df_pred.to_csv(output_file, index=False)

print(f"✓ Saved: {output_file}")
print(f"  Total predictions: {len(df_pred)}")
print(f"  Predicted machine: {(y_test_pred == 1).sum()} ({(y_test_pred == 1).sum()/len(y_test_pred)*100:.1f}%)")
print(f"  Predicted human: {(y_test_pred == 0).sum()} ({(y_test_pred == 0).sum()/len(y_test_pred)*100:.1f}%)")

## 5. Alternative: PCA + KNN

**PCA (Principal Component Analysis):**
- Reduces 5000 features to k components
- Keeps directions with highest variance
- **Key insight:** Fewer components can perform BETTER (curse of dimensionality)

**KNN (k-Nearest Neighbors):**
- Classifies by majority vote of k nearest training samples
- No model fitting, just distance computation
- With k=2, relies on 1-2 nearest samples

In [ ]:
print("Training PCA(100) + KNN(k=2) alternative...")

# PCA on training data
pca = PCA(n_components=100, svd_solver="randomized", random_state=SEED)
X_tr_pca = pca.fit_transform(X_tr)
X_val_pca = pca.transform(X_val)

print(f"PCA(100) explains {pca.explained_variance_ratio_.sum():.2%} of variance")

# KNN classifier
knn = KNeighborsClassifier(n_neighbors=2, n_jobs=-1)
knn.fit(X_tr_pca, y_tr)

y_val_pred_knn = knn.predict(X_val_pca)
f1_knn = f1_score(y_val, y_val_pred_knn, average="macro")
acc_knn = accuracy_score(y_val, y_val_pred_knn)

print(f"\nALTERNATIVE (PCA + KNN) Results:")
print(f"  Validation Macro F1: {f1_knn:.4f}")
print(f"  Validation Accuracy: {acc_knn:.4f}")

# Predict test set
pca_full = PCA(n_components=100, svd_solver="randomized", random_state=SEED)
X_train_pca = pca_full.fit_transform(X_feat_train)
X_test_pca = pca_full.transform(X_feat_test)

knn_final = KNeighborsClassifier(n_neighbors=2, n_jobs=-1)
knn_final.fit(X_train_pca, y_train)
y_test_pred_knn = knn_final.predict(X_test_pca)

output_file = OUTPUT_DIR / "Alternative_PCA_KNN_Prediction.csv"
df_pred_knn = pd.DataFrame({"id": test_ids, "label": y_test_pred_knn.astype(int)})
df_pred_knn.to_csv(output_file, index=False)
print(f"✓ Saved: {output_file}")

## 6. Summary & Comparison

In [ ]:
# Summary table
print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)

summary = pd.DataFrame([
    {"Model": "Baseline: LogReg on 5000 features", "Macro F1": f1_baseline, "Accuracy": acc_baseline},
    {"Model": "Alternative: PCA(100) + KNN(k=2)", "Macro F1": f1_knn, "Accuracy": acc_knn},
    {"Model": "🏆 WINNER: Custom TF-IDF + Linear SVM", "Macro F1": best_f1_winner, "Accuracy": None}
])

print(summary.to_string(index=False))

print(f"\nImprovements:")
print(f"  Winner vs Baseline: +{(best_f1_winner - f1_baseline):.4f} macro F1 (+{(best_f1_winner/f1_baseline - 1)*100:.1f}%)")
print(f"  Winner vs Alternative: +{(best_f1_winner - f1_knn):.4f} macro F1 (+{(best_f1_winner/f1_knn - 1)*100:.1f}%)")

print(f"\nPrediction files:")
for f in OUTPUT_DIR.glob("*.csv"):
    print(f"  ✓ {f.name}")

print("\n" + "="*70)

## Key Takeaways

1. **Representation >> Model:** Better features matter more than model complexity.
2. **Character n-grams = style fingerprint:** Machine text has different punctuation/spacing patterns.
3. **Balance matters:** With imbalanced data, use `class_weight` to weight minority class.
4. **Simpler wins:** Linear SVM beats complex ensembles when features are good.
5. **Validation = proxy for leaderboard:** Holdout ranking matched Kaggle ranking exactly.